In [28]:
import sys, pandas as pd
print(sys.executable)
print(pd.__version__)

/Users/mehalgosalia/anaconda3/envs/gibleed/bin/python
3.1.0


In [29]:
import os
directory = 'GiBleed_5.5'
print(sorted(os.listdir(directory)))

['CARE_SITE.csv', 'CDM_SOURCE.csv', 'COHORT.csv', 'COHORT_DEFINITION.csv', 'CONCEPT.csv', 'CONCEPT_ANCESTOR.csv', 'CONCEPT_CLASS.csv', 'CONCEPT_METADATA.csv', 'CONCEPT_RELATIONSHIP.csv', 'CONCEPT_RELATIONSHIP_METADATA.csv', 'CONCEPT_SYNONYM.csv', 'CONDITION_ERA.csv', 'CONDITION_OCCURRENCE.csv', 'COST.csv', 'DEATH.csv', 'DEVICE_EXPOSURE.csv', 'DOMAIN.csv', 'DOSE_ERA.csv', 'DRUG_ERA.csv', 'DRUG_EXPOSURE.csv', 'DRUG_STRENGTH.csv', 'EPISODE.csv', 'EPISODE_EVENT.csv', 'FACT_RELATIONSHIP.csv', 'LOCATION.csv', 'MEASUREMENT.csv', 'METADATA.csv', 'NOTE.csv', 'NOTE_NLP.csv', 'OBSERVATION.csv', 'OBSERVATION_PERIOD.csv', 'PACK_CONTENT.csv', 'PAYER_PLAN_PERIOD.csv', 'PERSON.csv', 'PROCEDURE_OCCURRENCE.csv', 'PROVIDER.csv', 'RELATIONSHIP.csv', 'SOURCE_TO_CONCEPT_MAP.csv', 'SPECIMEN.csv', 'VISIT_DETAIL.csv', 'VISIT_OCCURRENCE.csv', 'VOCABULARY.csv']


In [30]:
table_names = ['PERSON', 'OBSERVATION_PERIOD', 'DRUG_EXPOSURE',
               'CONDITION_OCCURRENCE', 'VISIT_OCCURRENCE', 'CONCEPT', 'CONCEPT_ANCESTOR']
tables = {}

for name in table_names:
    path = os.path.join(directory, name + ".csv")   # build the file name from `name`
    tables[name] = pd.read_csv(path)       # read the file at `path`
    print(name, len(tables[name]), list(tables[name].columns))              # number of rows, list of columns

PERSON 2694 ['PERSON_ID', 'GENDER_CONCEPT_ID', 'YEAR_OF_BIRTH', 'MONTH_OF_BIRTH', 'DAY_OF_BIRTH', 'BIRTH_DATETIME', 'RACE_CONCEPT_ID', 'ETHNICITY_CONCEPT_ID', 'LOCATION_ID', 'PROVIDER_ID', 'CARE_SITE_ID', 'PERSON_SOURCE_VALUE', 'GENDER_SOURCE_VALUE', 'GENDER_SOURCE_CONCEPT_ID', 'RACE_SOURCE_VALUE', 'RACE_SOURCE_CONCEPT_ID', 'ETHNICITY_SOURCE_VALUE', 'ETHNICITY_SOURCE_CONCEPT_ID']
OBSERVATION_PERIOD 5343 ['OBSERVATION_PERIOD_ID', 'PERSON_ID', 'OBSERVATION_PERIOD_START_DATE', 'OBSERVATION_PERIOD_END_DATE', 'PERIOD_TYPE_CONCEPT_ID']
DRUG_EXPOSURE 67707 ['DRUG_EXPOSURE_ID', 'PERSON_ID', 'DRUG_CONCEPT_ID', 'DRUG_EXPOSURE_START_DATE', 'DRUG_EXPOSURE_START_DATETIME', 'DRUG_EXPOSURE_END_DATE', 'DRUG_EXPOSURE_END_DATETIME', 'VERBATIM_END_DATE', 'DRUG_TYPE_CONCEPT_ID', 'STOP_REASON', 'REFILLS', 'QUANTITY', 'DAYS_SUPPLY', 'SIG', 'ROUTE_CONCEPT_ID', 'LOT_NUMBER', 'PROVIDER_ID', 'VISIT_OCCURRENCE_ID', 'VISIT_DETAIL_ID', 'DRUG_SOURCE_VALUE', 'DRUG_SOURCE_CONCEPT_ID', 'ROUTE_SOURCE_VALUE', 'DOSE_UNIT

In [31]:
import duckdb
person = tables["PERSON"]
obs_period = tables["OBSERVATION_PERIOD"]
duckdb.sql("SELECT COUNT (DISTINCT PERSON_ID) FROM person")
duckdb.sql("""
    SELECT n, COUNT(*) AS n_people
    FROM (
        SELECT PERSON_ID, COUNT(*) AS n
        FROM obs_period
        GROUP BY PERSON_ID
    ) AS t
    GROUP BY n
""")

nBIGINT,n_peopleBIGINT
1,5343


In [32]:
duckdb.sql("""
    SELECT
        CASE
            WHEN p.person_id IS NULL THEN 'obs_period_only'
            WHEN o.person_id IS NULL THEN 'person_only'
            ELSE 'both'
        END AS membership,
        COUNT(*) AS n
    FROM (SELECT DISTINCT person_id FROM person) AS p
    FULL OUTER JOIN
         (SELECT DISTINCT person_id FROM obs_period) AS o
      ON p.person_id = o.person_id
    GROUP BY membership
    ORDER BY membership
""")

membershipVARCHAR,nBIGINT
both,2694
obs_period_only,2649


In [33]:
drug_exp = tables["DRUG_EXPOSURE"]
cond_occ = tables["CONDITION_OCCURRENCE"]
duckdb.sql("""
    SELECT COUNT(DISTINCT person_id) AS n
    FROM drug_exp                                        
    WHERE person_id NOT IN (SELECT person_id FROM person)   
""")


nBIGINT
0


In [34]:
duckdb.sql("""
    SELECT COUNT(DISTINCT person_id) AS n
    FROM cond_occ                                       
    WHERE person_id NOT IN (SELECT person_id FROM person)   
""")

nBIGINT
0


## Step 2 — Concept sets
Find every standard concept for the exposures and the outcome using CONCEPT_ANCESTOR (ingredient/finding → all descendants), then check which of them actually appear in the patient data.

In [35]:
concept = tables["CONCEPT"]
concept_anc = tables["CONCEPT_ANCESTOR"]
concept_rel = pd.read_csv(os.path.join(directory, "CONCEPT_RELATIONSHIP.csv"))

In [36]:
# exposure concept set: all descendants of the two ingredients
drug_cs = duckdb.sql("""
    SELECT ca.ancestor_concept_id AS ingredient_id,
           c.concept_id, c.concept_name, c.concept_class_id, c.standard_concept
    FROM concept_anc AS ca
    JOIN concept AS c ON c.concept_id = ca.descendant_concept_id
    WHERE ca.ancestor_concept_id IN (1118084, 1124300)
    ORDER BY ingredient_id, c.concept_id
""").df()
drug_cs

,ingredient_id,CONCEPT_ID,CONCEPT_NAME,CONCEPT_CLASS_ID,STANDARD_CONCEPT
0,1118084,1118084,celecoxib,Ingredient,S
1,1124300,1124300,Diclofenac,Ingredient,S


In [37]:
# outcome concept set: all descendants of Gastrointestinal hemorrhage
gi_cs = duckdb.sql("""
    SELECT ca.ancestor_concept_id AS parent_id,
           c.concept_id, c.concept_name, c.concept_class_id, c.standard_concept
    FROM concept_anc AS ca
    JOIN concept AS c ON c.concept_id = ca.descendant_concept_id
    WHERE ca.ancestor_concept_id = 192671
    ORDER BY c.concept_id
""").df()
gi_cs

,parent_id,CONCEPT_ID,CONCEPT_NAME,CONCEPT_CLASS_ID,STANDARD_CONCEPT
0,192671,192671,Gastrointestinal hemorrhage,Clinical Finding,S


In [38]:
# coverage: which drug concepts are used in DRUG_EXPOSURE
duckdb.sql("""
    SELECT cs.ingredient_id, cs.concept_name, COUNT(*) AS n_rows,
           COUNT(DISTINCT d.person_id) AS n_people
    FROM drug_cs AS cs
    JOIN drug_exp AS d ON d.drug_concept_id = cs.concept_id
    GROUP BY ALL
    ORDER BY n_rows DESC
""")

ingredient_idBIGINT,CONCEPT_NAMEVARCHAR,n_rowsBIGINT,n_peopleBIGINT
1118084,celecoxib,1844,1844
1124300,Diclofenac,850,850


In [39]:
# coverage: which GI bleed concepts are used in CONDITION_OCCURRENCE
duckdb.sql("""
    SELECT cs.concept_name, COUNT(*) AS n_rows,
           COUNT(DISTINCT c.person_id) AS n_people
    FROM gi_cs AS cs
    JOIN cond_occ AS c ON c.condition_concept_id = cs.concept_id
    GROUP BY ALL
    ORDER BY n_rows DESC
""")

CONCEPT_NAMEVARCHAR,n_rowsBIGINT,n_peopleBIGINT
Gastrointestinal hemorrhage,479,479


In [40]:
# incomplete vocabulary: example formulations exist in CONCEPT but have no ancestor link to their ingredient
duckdb.sql("""
    SELECT c.concept_id, c.concept_name, c.concept_class_id,
           EXISTS (SELECT 1 FROM concept_anc AS ca
                   WHERE ca.descendant_concept_id = c.concept_id
                     AND ca.ancestor_concept_id IN (1118084, 1124300)) AS linked_to_ingredient
    FROM concept AS c
    WHERE c.concept_id IN (1118088, 40162359)
""")

CONCEPT_IDBIGINT,CONCEPT_NAMEVARCHAR,CONCEPT_CLASS_IDVARCHAR,linked_to_ingredientBOOLEAN
1118088,celecoxib 200 MG Oral Capsule [Celebrex],Branded Drug,False
40162359,Diclofenac Sodium 75 MG Delayed Release Oral Tablet,Clinical Drug,False


In [41]:
# source mapping (for understanding only): the ICD10CM code is non-standard and maps to SNOMED 192671
display(concept[concept["CONCEPT_ID"] == 35208414][["CONCEPT_ID", "CONCEPT_NAME", "VOCABULARY_ID", "STANDARD_CONCEPT"]])
display(concept_rel[(concept_rel["CONCEPT_ID_1"] == 35208414) & (concept_rel["RELATIONSHIP_ID"] == "Maps to")])
duckdb.sql("""
    SELECT condition_concept_id, condition_source_concept_id, COUNT(*) AS n_rows
    FROM cond_occ
    WHERE condition_concept_id = 192671
    GROUP BY ALL
""")

,CONCEPT_ID,CONCEPT_NAME,VOCABULARY_ID,STANDARD_CONCEPT
0,35208414,"Gastrointestinal hemorrhage, unspecified",ICD10CM,NaN


,CONCEPT_ID_1,CONCEPT_ID_2,RELATIONSHIP_ID,VALID_START_DATE,VALID_END_DATE,INVALID_REASON
3,35208414,192671,Maps to,1970-01-01,2099-12-31,NaN


CONDITION_CONCEPT_IDBIGINT,CONDITION_SOURCE_CONCEPT_IDBIGINT,n_rowsBIGINT
192671,35208414,479


In [42]:
# records per person per drug, and patients with both drugs
display(duckdb.sql("""
    SELECT drug_concept_id, MAX(n) AS max_rows_per_person
    FROM (SELECT person_id, drug_concept_id, COUNT(*) AS n
          FROM drug_exp
          WHERE drug_concept_id IN (1118084, 1124300)
          GROUP BY ALL)
    GROUP BY ALL
"""))
duckdb.sql("""
    SELECT COUNT(*) AS n_both_drugs
    FROM (SELECT person_id
          FROM drug_exp
          WHERE drug_concept_id IN (1118084, 1124300)
          GROUP BY person_id
          HAVING COUNT(DISTINCT drug_concept_id) = 2)
""")

DRUG_CONCEPT_IDBIGINT,max_rows_per_personBIGINT
1124300,1
1118084,1


n_both_drugsBIGINT
0


In [43]:
# final concept-set table (as required by the brief)
concept_set_table = pd.DataFrame([
    (1118084, "celecoxib", "Exposure: treatment = 1", "Ingredient + all CONCEPT_ANCESTOR descendants via DRUG_CONCEPT_ID"),
    (1124300, "Diclofenac", "Exposure: comparator = 0", "Ingredient + all CONCEPT_ANCESTOR descendants via DRUG_CONCEPT_ID"),
    (192671, "Gastrointestinal hemorrhage", "Outcome / baseline exclusion", "Standard SNOMED concept + descendants via CONDITION_CONCEPT_ID"),
    (35208414, "Gastrointestinal hemorrhage, unspecified (ICD10CM)", "Source code: mapping check only", "Not used for selection; maps to 192671"),
], columns=["concept_id", "concept_name", "role", "inclusion_rule"])
concept_set_table

,concept_id,concept_name,role,inclusion_rule
0,1118084,celecoxib,Exposure: treatment = 1,Ingredient + all CONCEPT_ANCESTOR descendants ...
1,1124300,Diclofenac,Exposure: comparator = 0,Ingredient + all CONCEPT_ANCESTOR descendants ...
2,192671,Gastrointestinal hemorrhage,Outcome / baseline exclusion,Standard SNOMED concept + descendants via COND...
3,35208414,"Gastrointestinal hemorrhage, unspecified (ICD1...",Source code: mapping check only,Not used for selection; maps to 192671
